> ⚠️ **This notebook requires IBM Quantum credentials and is not part of the simulation results.**

# Notebook 04 — Legacy Hardware Execution (DEPRECATED)

> :warning: **THIS NOTEBOOK IS DEPRECATED. DO NOT RUN ON HARDWARE.**
> The correct hardware-execution pipeline is in **Notebook 10**.
> This notebook is preserved only as a record of the *incorrect* approach we first tried, and why it failed.

## What this notebook used to do (wrong)

It ran a **full VQE optimisation loop** (300 COBYLA iterations) directly on IBM hardware. Each iteration submitted a fresh `EstimatorV2` job, requiring a long-lived `Session`.

## Why it failed

1. **IBM Open Plan session limit.** Sessions on the Open plan are capped at **10 minutes** of total runtime. 300 COBYLA iterations with 2048 shots and a transpiled 12-qubit ansatz take ~6 minutes per *single* expectation value. The session times out at step ~2.
2. **Open Plan does not allow Session mode at all** (HTTP 400, error 1352). Sessions require an Hourly Premium plan. The Open plan only supports `Batch` and standalone job execution.
3. **Noise-driven optimiser collapse.** Even when sessions were available (Standard plan), COBYLA cannot tolerate the ~50 mHa shot-noise variance on hardware - the optimiser wanders aimlessly. Reported convergence rates were <5% across 40 attempts.
4. **Cost.** A single 300-iteration hardware VQE run consumes ~6,000 shots * 300 iter = 1.8 million shots, well beyond any free-tier credit.

## The correct architecture (-> Notebook 10)

Do the optimisation **classically** on a statevector simulator, save theta\*, **bind** the ansatz, **transpile once**, and submit a **single** EstimatorV2 PUB.


In [ ]:
# DO NOT RUN. Preserved only for reference / SI of the manuscript.
raise RuntimeError(
    'Notebook 04 is the deprecated legacy hardware loop. '
    'Use Notebook 10 (single-PUB pipeline) instead.'
)

# What follows would be the legacy code (commented out):
# from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2 as Estimator
# from qiskit.circuit.library import EfficientSU2
# from scipy.optimize import minimize
#
# service = QiskitRuntimeService(channel='ibm_quantum_platform', token=YOUR_TOKEN)
# backend = service.least_busy(operational=True, simulator=False)
# ansatz  = EfficientSU2(12, reps=4, entanglement='full')
#
# def hardware_cost(params, ansatz=ansatz, qubit_op=qubit_op):
#     # This requires a new Estimator.run() per call -> 300 calls -> session timeout
#     with Session(backend=backend) as session:  # FAILS on Open Plan
#         estimator = Estimator(mode=session)
#         job = estimator.run([(ansatz.assign_parameters(params), qubit_op)])
#         return job.result()[0].data.evs.real
#
# x0 = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
# res = minimize(hardware_cost, x0, method='COBYLA', options={'maxiter': 300, 'rhobeg': 0.3})
#
# # Above will fail because:
# # - On Open Plan: HTTP 400 'You are not authorized to run a session when using the open plan.'
# # - On Standard Plan: 300 iters * 30 s/iter = 150 min, well past the 10 min hard cap on Open
# # - Noise variance ~50 mHa swamps the optimiser landscape (Powell, J. Comput. Phys. 1965)


## See also

- **Notebook 10** — correct hardware pipeline (single Batch + single PUB)
- **Notebook 09** — classical reference and the frozen-core bug fix
- `results/hardware_job_id.txt` — IBM Quantum job ID from the verified run
